# 07 - Evaluation

## Overview

Evaluate centralized XGBoost & LightGBM on test set with full metrics.
Also loads FL training history from notebook 06 for comparison.

In [ ]:
# Imports
import numpy as np
import pandas as pd
import os
import json
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '/home/willtanoe/Documents/fl-xgb-thesis')
from src.config import (
    NUM_CLIENTS, FL_ROUNDS, SEED, CPU_THREADS,
    XGB_PARAMS, LGB_PARAMS
)

import joblib
import xgboost as xgb
import lightgbm as lgb

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)

# Paths
PREPROCESSED_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/preprocessed"
LOGS_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/logs"
FIGURES_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Notebook 07 - Evaluation initialized.")

## 1. Load Data

In [ ]:
print("Loading data...")
train_df = pd.read_parquet(os.path.join(PREPROCESSED_DIR, "train.parquet"))
test_df = pd.read_parquet(os.path.join(PREPROCESSED_DIR, "test.parquet"))

with open(os.path.join(PREPROCESSED_DIR, "metadata.json"), 'r') as f:
    metadata = json.load(f)

feature_cols = metadata['feature_cols']
num_classes = metadata['num_classes']

le = joblib.load(os.path.join(PREPROCESSED_DIR, "label_encoder.pkl"))
class_names = le.classes_

X_train = train_df[feature_cols].values
y_train = train_df['label'].values
X_test = test_df[feature_cols].values
y_test = test_df['label'].values

n_jobs = CPU_THREADS  # match flwr_app setting (can override to 1 if contention)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Classes: {num_classes}")
print(f"Label encoder fitted on: {len(class_names)} classes")

## 2. Load FL Training History

In [ ]:
xgb_history_path = os.path.join(LOGS_DIR, "fl_history_xgb.json")
lgb_history_path = os.path.join(LOGS_DIR, "fl_history_lgb.json")

xgb_fl_data = {}
lgb_fl_data = {}

if os.path.exists(xgb_history_path):
    with open(xgb_history_path) as f:
        xgb_fl_data = json.load(f)
    print(f"Loaded XGB FL history: {len(xgb_fl_data.get('rounds', []))} rounds")

if os.path.exists(lgb_history_path):
    with open(lgb_history_path) as f:
        lgb_fl_data = json.load(f)
    print(f"Loaded LGB FL history: {len(lgb_fl_data.get('rounds', []))} rounds")

if not xgb_fl_data and not lgb_fl_data:
    print("No FL history found. Run notebook 06 first.")

## 3. Train Centralized Models

In [ ]:
MODELS_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/models"
xgb_path = os.path.join(MODELS_DIR, "centralized_xgboost.ubj")
lgb_path = os.path.join(MODELS_DIR, "centralized_lightgbm.txt")

print("Loading centralized XGBoost...")
xgb_clf = xgb.XGBClassifier()
xgb_clf.load_model(xgb_path)
print(f"XGBoost loaded ({xgb_path}).")

print("\nLoading centralized LightGBM...")
lgb_clf = lgb.Booster(model_file=lgb_path)
print(f"LightGBM loaded ({lgb_path}).")

## 4. Evaluate XGBoost

In [ ]:
print("\n" + "=" * 60)
print("XGBOOST EVALUATION")
print("=" * 60)

y_pred_xgb = xgb_clf.predict(X_test)

xgb_f1_macro = f1_score(y_test, y_pred_xgb, average='macro')
xgb_f1_weighted = f1_score(y_test, y_pred_xgb, average='weighted')
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)

print(f"\nXGBoost Test Metrics:")
print(f"  Accuracy:      {xgb_accuracy:.4f}")
print(f"  F1 Macro:      {xgb_f1_macro:.4f}")
print(f"  F1 Weighted:   {xgb_f1_weighted:.4f}")

print("\nPer-class F1 (XGBoost):")
xgb_f1_per_class = f1_score(y_test, y_pred_xgb, average=None)
per_class_df = pd.DataFrame({
    'class_id': range(num_classes),
    'class_name': class_names,
    'f1_score': xgb_f1_per_class
}).sort_values('f1_score', ascending=False)
print(per_class_df.to_string(index=False))

## 5. Evaluate LightGBM

In [ ]:
print("\n" + "=" * 60)
print("LIGHTGBM EVALUATION")
print("=" * 60)

y_pred_lgb = lgb_clf.predict(X_test).argmax(axis=1)

lgb_f1_macro = f1_score(y_test, y_pred_lgb, average='macro')
lgb_f1_weighted = f1_score(y_test, y_pred_lgb, average='weighted')
lgb_accuracy = accuracy_score(y_test, y_pred_lgb)

print(f"\nLightGBM Test Metrics:")
print(f"  Accuracy:      {lgb_accuracy:.4f}")
print(f"  F1 Macro:      {lgb_f1_macro:.4f}")
print(f"  F1 Weighted:   {lgb_f1_weighted:.4f}")

print("\nPer-class F1 (LightGBM):")
lgb_f1_per_class = f1_score(y_test, y_pred_lgb, average=None)
per_class_df_lgb = pd.DataFrame({
    'class_id': range(num_classes),
    'class_name': class_names,
    'f1_score': lgb_f1_per_class
}).sort_values('f1_score', ascending=False)
print(per_class_df_lgb.to_string(index=False))

## 6. Model Comparison

In [ ]:
print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

comparison = pd.DataFrame({
    'Model': ['XGBoost', 'LightGBM', 'FL XGBoost', 'FL LightGBM'],
    'Accuracy': [xgb_accuracy, lgb_accuracy, fl_xgb_acc, fl_lgb_acc],
    'F1 Macro': [xgb_f1_macro, lgb_f1_macro, fl_xgb_f1, fl_lgb_f1],
    'F1 Weighted': [xgb_f1_weighted, lgb_f1_weighted]
})
print(comparison.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

models = ['XGBoost', 'LightGBM']
metrics_dict = {
    'Accuracy': [xgb_accuracy, lgb_accuracy],
    'F1 Macro': [xgb_f1_macro, lgb_f1_macro],
    'F1 Weighted': [xgb_f1_weighted, lgb_f1_weighted]
}

x = np.arange(len(models))
width = 0.25

for i, (metric_name, values) in enumerate(metrics_dict.items()):
    bars = ax.bar(x + i * width, values, width, label=metric_name)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Score')
ax.set_title('Centralized Model Comparison on Test Set')
ax.set_xticks(x + width)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '07_centralized_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: results/figures/07_centralized_comparison.png")

## 7. FL Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if xgb_fl_data.get('f1_macro'):
    rounds = xgb_fl_data['rounds']
    total_trees = [r * xgb_fl_data.get('trees_per_round', 20) for r in rounds]
    axes[0].plot(total_trees, xgb_fl_data['f1_macro'], 'b-', label='FL XGBoost F1')
    axes[0].plot(total_trees, xgb_fl_data['accuracy'], 'b--', label='FL XGBoost Acc')
axes[0].axhline(y=xgb_f1_macro, color='b', linestyle=':', alpha=0.5, label=f'Centralized F1={xgb_f1_macro:.3f}')
axes[0].set_xlabel('Total Trees (accumulated)')
axes[0].set_ylabel('Score')
axes[0].set_title('XGBoost: FL vs Centralized')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

if lgb_fl_data.get('f1_macro'):
    rounds = lgb_fl_data['rounds']
    total_trees = [r * lgb_fl_data.get('trees_per_round', 20) for r in rounds]
    axes[1].plot(total_trees, lgb_fl_data['f1_macro'], 'orange', label='FL LightGBM F1')
    axes[1].plot(total_trees, lgb_fl_data['accuracy'], 'orange', ls='--', label='FL LightGBM Acc')
axes[1].axhline(y=lgb_f1_macro, color='orange', linestyle=':', alpha=0.5, label=f'Centralized F1={lgb_f1_macro:.3f}')
axes[1].set_xlabel('Total Trees (accumulated)')
axes[1].set_ylabel('Score')
axes[1].set_title('LightGBM: FL vs Centralized')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '07_fl_vs_centralized.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: results/figures/07_fl_vs_centralized.png")

## FL Model Evaluation

Load FL trained models from `flwr run` and evaluate on full test set.


In [ ]:
MODELS_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/models"

# Load FL XGBoost
fl_xgb_path = os.path.join(MODELS_DIR, 'fl_xgboost.ubj')
fl_lgb_path = os.path.join(MODELS_DIR, 'fl_lightgbm.txt')

fl_xgb_pred = None
fl_lgb_pred = None

if os.path.exists(fl_xgb_path):
    fl_xgb = xgb.XGBClassifier()
    fl_xgb.load_model(fl_xgb_path)
    fl_xgb_pred = fl_xgb.predict(X_test)
    fl_xgb_acc = accuracy_score(y_test, fl_xgb_pred)
    fl_xgb_f1 = f1_score(y_test, fl_xgb_pred, average='macro')
    print(f"FL XGBoost: Acc={fl_xgb_acc:.4f}, F1={fl_xgb_f1:.4f}")
else:
    print("FL XGBoost model not found. Run notebook 06 first.")

if os.path.exists(fl_lgb_path):
    fl_lgb = lgb.Booster(model_file=fl_lgb_path)
    fl_lgb_pred = fl_lgb.predict(X_test).argmax(axis=1)
    fl_lgb_acc = accuracy_score(y_test, fl_lgb_pred)
    fl_lgb_f1 = f1_score(y_test, fl_lgb_pred, average='macro')
    print(f"FL LightGBM: Acc={fl_lgb_acc:.4f}, F1={fl_lgb_f1:.4f}")
else:
    print("FL LightGBM model not found. Run notebook 06 first.")


## 9. Confusion Matrix (Normalized)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

for idx, (model_name, y_pred, cmap) in enumerate([
    ('XGBoost', y_pred_xgb, 'Blues'),
    ('LightGBM', y_pred_lgb, 'Oranges'),
]):
    ax = axes[idx]
    cm = confusion_matrix(y_test, y_pred, normalize='true')
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, include_values=True, values_format='.2f', cmap=cmap, xticks_rotation=90, colorbar=True, text_kw={'fontsize': 5})
    ax.set_title(f'{model_name} Confusion Matrix (normalized by true label)')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

fig.subplots_adjust(bottom=0.2)
plt.savefig(os.path.join(FIGURES_DIR, '07_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: results/figures/07_confusion_matrix.png")

# Print numeric summary
print("\nConfusion Matrix Numeric Summary (normalized by true label):")
print(f"{'Class':<30} {'XGB Recall':>12} {'LGB Recall':>12} {'XGB F1':>10} {'LGB F1':>10}")
print("-" * 74)
xgb_prec = precision_score(y_test, y_pred_xgb, average=None, zero_division=0)
lgb_prec = precision_score(y_test, y_pred_lgb, average=None, zero_division=0)
xgb_rec = recall_score(y_test, y_pred_xgb, average=None, zero_division=0)
lgb_rec = recall_score(y_test, y_pred_lgb, average=None, zero_division=0)
for i, name in enumerate(class_names):
    xgb_f1 = 2 * xgb_prec[i] * xgb_rec[i] / (xgb_prec[i] + xgb_rec[i]) if (xgb_prec[i] + xgb_rec[i]) > 0 else 0.0
    lgb_f1 = 2 * lgb_prec[i] * lgb_rec[i] / (lgb_prec[i] + lgb_rec[i]) if (lgb_prec[i] + lgb_rec[i]) > 0 else 0.0
    print(f'{name:<30} {xgb_rec[i]:>12.4f} {lgb_rec[i]:>12.4f} {xgb_f1:>10.4f} {lgb_f1:>10.4f}')
print("-" * 74)
print(f'{"Macro Average":<30} {np.mean(xgb_rec):>12.4f} {np.mean(lgb_rec):>12.4f} '
      f'{f1_score(y_test, y_pred_xgb, average="macro", zero_division=0):>10.4f} '
      f'{f1_score(y_test, y_pred_lgb, average="macro", zero_division=0):>10.4f}')

## 10. Per-Class F1 Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

x = np.arange(num_classes)
width = 0.35

ax.barh(x - width/2, xgb_f1_per_class, width, label='XGBoost', alpha=0.8)
ax.barh(x + width/2, lgb_f1_per_class, width, label='LightGBM', alpha=0.8)

ax.set_xlabel('F1 Score')
ax.set_ylabel('Class ID')
ax.set_title('Per-Class F1 Score Comparison (Centralized)')
ax.set_yticks(x)
ax.set_yticklabels([f"{i}: {name[:20]}" for i, name in enumerate(class_names)])
ax.legend()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '07_per_class_f1.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: results/figures/07_per_class_f1.png")

## 11. Save Evaluation Results

In [ ]:
results = {
    'centralized': {
        'xgboost': {
            'accuracy': float(xgb_accuracy),
            'f1_macro': float(xgb_f1_macro),
            'f1_weighted': float(xgb_f1_weighted),
            'f1_per_class': {cls: float(f1) for cls, f1 in zip(class_names, xgb_f1_per_class)}
        },
        'lightgbm': {
            'accuracy': float(lgb_accuracy),
            'f1_macro': float(lgb_f1_macro),
            'f1_weighted': float(lgb_f1_weighted),
            'f1_per_class': {cls: float(f1) for cls, f1 in zip(class_names, lgb_f1_per_class)}
        },
    },
    'fl_history': {
        'xgboost': xgb_fl_data,
        'lightgbm': lgb_fl_data,
    },
    'fl': {
        'xgboost': {
            'accuracy': float(fl_xgb_acc) if os.path.exists(fl_xgb_path) else None,
            'f1_macro': float(fl_xgb_f1) if os.path.exists(fl_xgb_path) else None
        },
        'lightgbm': {
            'accuracy': float(fl_lgb_acc) if os.path.exists(fl_lgb_path) else None,
            'f1_macro': float(fl_lgb_f1) if os.path.exists(fl_lgb_path) else None
        }
    },
    'test_samples': int(len(y_test)),
    'num_classes': num_classes,
}

with open(os.path.join(LOGS_DIR, 'evaluation_results.json'), 'w') as f:
    json.dump(results, f, indent=2)
print("Evaluation results saved: results/logs/evaluation_results.json")

## 12. Summary

In [ ]:
print("\n" + "=" * 60)
print("FINAL EVALUATION SUMMARY")
print("=" * 60)

print("\nTest Set Results:")
print(f"""
+------------+-----------+-----------+---------------+
| Model      | Accuracy  | F1 Macro  | F1 Weighted   |
+------------+-----------+-----------+---------------+
| XGBoost    | {xgb_accuracy:.4f}   | {xgb_f1_macro:.4f}   | {xgb_f1_weighted:.4f}       |
| LightGBM   | {lgb_accuracy:.4f}   | {lgb_f1_macro:.4f}   | {lgb_f1_weighted:.4f}       |
+------------+-----------+-----------+---------------+
| FL XGBoost | {fl_xgb_acc:.4f}   | {fl_xgb_f1:.4f}   |       N/A       |
| FL LightGBM| {fl_lgb_acc:.4f}   | {fl_lgb_f1:.4f}   |       N/A       |
+------------+-----------+-----------+---------------+
""")

print("\nOutput files:")
print(f"  - results/logs/evaluation_results.json")
print(f"  - results/figures/07_centralized_comparison.png")
print(f"  - results/figures/07_confusion_matrix.png")
print(f"  - results/figures/07_per_class_f1.png")
print(f"  - results/figures/07_fl_vs_centralized.png")

print("\nAll notebooks completed!")